# 00 · Setup
Verifică runtime-ul (Python, GPU), instalează pachetele, conectează Earth Engine și confirmă zona.
Rulează-l o dată la fiecare runtime nou.

In [ ]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul gisday2026): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/gisday2026.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/gisday2026'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/gisday2026'], check=True)
        os.chdir('/content/gisday2026')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

In [ ]:
import platform
print('Python', platform.python_version())
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or 'Fără GPU vizibil')

In [ ]:
%pip install -q earthengine-api geopandas pyarrow rasterio rasterstats lightgbm shap pystac-client planetary-computer matplotlib

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

In [ ]:
print('Suprafața zonei (km²):', round(AOI.area(10).divide(1e6).getInfo(), 1))
try:
    import torch
    print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except ImportError:
    print('torch nu e instalat pe acest kernel')